In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

%load_ext autoreload
%autoreload 2

sys.path.append(str(Path.cwd().parent / "scripts"))
from style import set_default_style, set_publication_style
from grna_assignment import assign_grna_to_cell

set_default_style()

In [ ]:
XENIUM_PATHS = [
    '../data/lipogrid/pilot/coregistration/output-XETG00334__0058728__0058728__20250605__132649_count_matrix_watershed_expanded30_hood.h5ad',
    '../data/lipogrid/pilot/coregistration/output-XETG00334__0058741__0058741__20250605__132650_count_matrix_watershed_expanded30_hood.h5ad',
    '../data/lipogrid/pilot/coregistration/output-XETG00075__0054781__Lipo__20251216__164421_count_matrix_Xenium_segmentation_expanded30.h5ad',
    '../data/lipogrid/pilot/coregistration/output-XETG00075__0064773__Lipo__20251216__164422_count_matrix_Xenium_segmentation_expanded30.h5ad',
]

# watershed-expanded Xenium count matrices (nuclear DAPI signal expanded to approximate cell bounds)
adatas, dfs = [], []
for path in XENIUM_PATHS:
    adata = ad.read_h5ad(path)
    adata.obs['cells'] = adata.obs_names
    df = pd.DataFrame(adata.X, index=adata.obs['cells'], columns=adata.var_names)
    df = df.loc[:, df.columns.str.contains('gRNA')].copy()  # drop non-gRNA probes
    adatas.append(adata)
    dfs.append(df)

dfs[0]

In [ ]:
cell_grnas = [assign_grna_to_cell(df) for df in dfs]

In [ ]:
for i, cell_grna in enumerate(cell_grnas, start=1):
    counts = pd.Series(cell_grna).value_counts()
    n_single = counts.drop(index=['no_gRNA', 'low_count', 'multiple_gRNAs', 'ambiguous'], errors='ignore').sum()
    print(f"Slide {i}")
    print(f"  no gRNA:        {counts.get('no_gRNA', 0)}")
    print(f"  low count:      {counts.get('low_count', 0)}")
    print(f"  multiple gRNAs: {counts.get('multiple_gRNAs', 0)}")
    print(f"  ambiguous:      {counts.get('ambiguous', 0)}")
    print(f"  single gRNA:    {n_single}")

In [ ]:
# strip the '.gRNAx' suffix to get target-gene-level counts per slide
cell_gene = [
    {cell: grna.split('.')[0] if isinstance(grna, str) else grna for cell, grna in cell_grna.items()}
    for cell_grna in cell_grnas
]
gene_counts = [pd.Series(g).value_counts() for g in cell_gene]
for i, counts in enumerate(gene_counts, start=1):
    print(f"Slide {i}: {len(counts)} unique gRNA targets (incl. Intergenic and QC categories)")
    print(counts)

In [ ]:
# stacked bar plot of gRNA assignments across slides, sorted by total count
def clean_grna_assignments(gene_map):
    return pd.Series(gene_map).replace(
        [None, 'low_count', 'multiple_gRNAs', 'ambiguous', 'Intergenic', 'no_gRNA'], pd.NA
    ).dropna()

slide_names = [f'Slide {i}' for i in range(1, 5)]
slide_counts = [clean_grna_assignments(g).value_counts() for g in cell_gene]
all_genes = sorted(set().union(*[c.index for c in slide_counts]))

data = pd.DataFrame({name: counts.reindex(all_genes, fill_value=0)
                     for name, counts in zip(slide_names, slide_counts)})
data = data.loc[data.sum(axis=1).sort_values(ascending=False).index]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
ax = data.plot(kind='bar', stacked=True, color=colors, figsize=(16, 8))
plt.title('Stacked distribution of gRNA assignments across all slides', fontsize=28, fontweight='bold')
plt.xlabel('gRNA target gene', fontsize=24, fontweight='bold')
plt.ylabel('Number of cells assigned', fontsize=24, fontweight='bold')
plt.xticks(rotation=90, visible=False, fontsize=22, fontweight='bold')
plt.yticks(fontsize=22, fontweight='bold')
plt.legend(fontsize=22)
plt.tight_layout()
plt.show()

In [ ]:
# PSMB3 and SNRNP200 knockouts are expected to be lethal -> should be rare but not absent
for gene in ['PSMB3', 'SNRNP200']:
    counts = [int(gc.get(gene, 0)) for gc in gene_counts]
    print(f"{gene}: {counts} across slides 1-4")

In [ ]:
for adata, cell_grna in zip(adatas, cell_grnas):
    adata.obs['gRNA'] = adata.obs['cells'].map(cell_grna)

adatas[3].obs['gRNA']

In [ ]:
# save anndata files with the gRNA assignments
OUT_PATHS = [
    '../data/lipogrid/pilot/coregistration/xenium_david_1_with_grna_132649.h5ad',
    '../data/lipogrid/pilot/coregistration/xenium_david_2_with_grna_132650.h5ad',
    '../data/lipogrid/pilot/coregistration/xenium_david_3_with_grna_164421.h5ad',
    '../data/lipogrid/pilot/coregistration/xenium_david_4_with_grna_164422.h5ad',
]
for adata, path in zip(adatas, OUT_PATHS):
    adata.write(path)

In [ ]:
## plot overall dominant gRNA distribution per cell
onedf = pd.concat(dfs)
set_publication_style()

# input: rows = cells, columns = gRNAs, values = UMI counts
umis      = onedf.to_numpy(dtype=float)
total_umi = umis.sum(axis=1)
dominant  = umis.max(axis=1)
frac_dom  = np.divide(dominant, total_umi,
                      out=np.full_like(total_umi, np.nan),
                      where=total_umi > 0)

diag = pd.DataFrame({
    "frac_dominant": frac_dom,
    "total_umi": total_umi,
    "dominant_umi": dominant,
}, index=onedf.index)

# filter: keep only cells with >= 8 UMIs of the dominant gRNA
diag = diag[diag["dominant_umi"] >= 8]
print(f"Cells retained (dominant gRNA >= 8 UMIs): {len(diag)}")

x  = diag["frac_dominant"].to_numpy()
y  = diag["total_umi"].to_numpy()
ly = np.log10(y)

# plot
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
hb = ax.hexbin(x, ly, gridsize=60, cmap="viridis", mincnt=2)

# relabel the (log10) y-axis with real UMI values
ticks = np.arange(0, np.ceil(ly.max()) + 1)          # start ticks at 10^0
ax.set_yticks(ticks)
ax.set_yticklabels([f"$10^{{{int(t)}}}$" for t in ticks])
ax.set_ylim(0, ly.max() + 0.05)    

ax.set_xlim(0, 1.02)
ax.axvline(0.5, color="grey", ls="--", lw=1)
ax.set_xlabel("Fraction of dominant gRNA")
ax.set_ylabel("Total gRNA UMIs per cell")
ax.tick_params(axis="x", length=5, width=1)
ax.tick_params(axis="y", length=5, width=1)

cbar = fig.colorbar(hb)
cbar.set_label("cells per bin")
cbar.outline.set_linewidth(1)
cbar.ax.tick_params(length=5, width=1)

fig.tight_layout()
fig.savefig(
    "../data/lipogrid/pilot/analysis/"
    "internalnorm_finalfigs/hexbin_dominantfrac_totalUMIwider.pdf",
    dpi=300, bbox_inches="tight",
)
plt.show()